# DGTW for funds: is it picking, timing, or style?

**Question.** Two large-cap value funds. Vanguard's Value ETF (VTV) is an index fund: it buys a published list of stocks. The MFS Value Fund (Class I, MEIIX) is actively managed: its managers choose what to own. Did either earn more than its stocks' *style* explains?

We use the Daniel, Grinblatt, Titman and Wermers (1997) approach from the Session 2 slides. Every stock is compared with a benchmark portfolio of look-alike stocks (one of 125 cells: size x book-to-market x momentum). A fund's monthly return on its stock holdings then splits into three parts:

| Part | Formula | Question |
|---|---|---|
| **CS**, characteristic selectivity | sum of w<sub>j</sub> (R<sub>j</sub> - R<sup>cell now</sup>) | Did the stocks beat their own look-alikes? |
| **CT**, characteristic timing | sum of w<sub>j</sub> R<sup>cell now</sup> - sum of w'<sub>j</sub> R<sup>cell a year ago</sup> | Did moving between styles over the year pay off? |
| **AS**, average style | sum of w'<sub>j</sub> R<sup>cell a year ago</sup> | What did the fund's usual style earn? |

w<sub>j</sub> are this month's weights, w'<sub>j</sub> the weights a year earlier. CS + CT + AS = the return on the fund's stock holdings, before fees and trading costs.

| Part | What you do |
|---|---|
| 1 | Look at the holdings, and at how much of each fund the benchmark covers |
| 2 | The characteristics view: is VTV a value fund by what it owns? |
| 3 | Compute CS, CT and AS every month |
| 4 | Read the results: skill, luck, or noise? |
| 5 | Compare the holdings return with the fund's actual return |
| 6 | Discuss |

Data: CRSP Mutual Fund Database holdings and returns, and DGTW benchmarks built from CRSP and Compustat (class use only).

> **A note on how this notebook was made.** It was written by an AI (Claude, from Anthropic), working from your instructor's lecture plan and checked against the data. Treat it the way you would treat any code you did not write: run it, read it, and check its claims against the output you see. If something looks wrong, tell your instructor.

## Part 0. Setup

In [ ]:
import numpy as np, pandas as pd, statsmodels.api as sm
import matplotlib.pyplot as plt
from pathlib import Path
pd.set_option("display.width", 160, "display.max_columns", 30, "display.float_format", "{:,.4f}".format)

# The class data (fund_data.zip). The link is on the course notes page, not in this public notebook.
DATA_ZIP_LINK = ""     # <- paste the Google Drive link to fund_data.zip between the quotes, then run this cell

import re, zipfile, urllib.request
DATA_DIR = Path("data")
if not (DATA_DIR / "fund_holdings.parquet").exists():
    fid = re.search(r"/d/([\w-]+)|[?&]id=([\w-]+)", DATA_ZIP_LINK.strip())
    assert fid, "Paste the fund_data.zip link from the course notes page into DATA_ZIP_LINK above, then rerun this cell."
    fid = fid.group(1) or fid.group(2)
    urllib.request.urlretrieve(f"https://drive.usercontent.google.com/download?id={fid}&export=download&confirm=t", "fund_data.zip")
    assert zipfile.is_zipfile("fund_data.zip"), "The download did not return a zip. Check that the link is shared with you, and try again."
    zipfile.ZipFile("fund_data.zip").extractall(".")          # creates ./data/
    print("downloaded and unzipped fund_data.zip")

H = pd.read_parquet(DATA_DIR / "fund_holdings.parquet")   # one row per fund, report date, security
D = pd.read_parquet(DATA_DIR / "dgtw_stocks.parquet")     # one row per stock and month: return, cell, cell return
R = pd.read_parquet(DATA_DIR / "fund_returns.parquet")    # each fund's actual monthly return
info = pd.read_csv(DATA_DIR / "fund_info.csv").set_index("fund")

FUNDS = ["VTV", "MFS"]
START, END = "2017-01", "2026-09"
print(info[["name", "share_class_ticker", "exp_ratio"]].to_string())
print(f"\nholdings: {H.report_dt.min().date()} to {H.report_dt.max().date()} | DGTW: {D.month.min()} to {D.month.max()}")

In [ ]:
def nw_mean(x, lags=3):
    """Average of a series and its Newey-West t-statistic (robust to autocorrelation)."""
    x = np.asarray(x, float)
    fit = sm.OLS(x, np.ones(len(x))).fit(cov_type="HAC", cov_kwds={"maxlags": lags})
    return fit.params[0], fit.tvalues[0]

## Part 1. The holdings

Funds report their holdings to data vendors on a *report date*. CRSP has these reports roughly every month for both funds since 2016. (VTV's reports in CRSP were only quarterly before about 2013: monthly data are a recent luxury.)

In [ ]:
H.groupby([H.fund, H.report_dt.dt.year])["report_dt"].nunique().unstack(0).T

In [ ]:
# The largest positions in each fund's latest report
for f in FUNDS:
    last = H.loc[H.fund == f, "report_dt"].max()
    h = H[(H.fund == f) & (H.report_dt == last)]
    print(f"{f}: report of {last.date()}, {len(h)} positions, top 10 = {h.weight.nlargest(10).sum():.0%} of assets")
    print(h.nlargest(10, "weight")[["security_name", "ticker", "weight"]].to_string(index=False), "\n")

**Read it.** VTV holds a few hundred stocks; MFS about 70. Which fund is more concentrated? Do the two funds share top holdings?

### 1.2 How much of each fund does the benchmark cover?

A holding gets a DGTW benchmark only if it is in the DGTW universe that month. Three things keep a holding out:

1. **No CRSP stock id (PERMNO):** cash, money-market funds, futures, foreign shares.
2. **Not a U.S.-incorporated common stock from a corporate issuer:** U.S.-listed companies incorporated in Ireland, Switzerland or the Netherlands, and REITs.
3. **No book-to-market:** firms with negative book equity (often after years of large buybacks) cannot be sorted on B/M.

In [ ]:
def coverage(fund, report_dt):
    h = H[(H.fund == fund) & (H.report_dt == report_dt)].copy()
    month = str(report_dt.to_period("M"))
    h["status"] = np.where(h.permno.isna(), "no PERMNO",
                  np.where(h.permno.isin(D.loc[D.month == month, "permno"]), "has a DGTW benchmark", "PERMNO, but not in DGTW"))
    return h

for f in FUNDS:
    h = coverage(f, H.loc[H.fund == f, "report_dt"].max())
    print(f, (h.groupby("status").weight.sum() * 100).round(1).to_dict())
    print(h[h.status != "has a DGTW benchmark"].nlargest(8, "weight")[["security_name", "status", "weight"]].to_string(index=False), "\n")

**How to read the output (our run, May 2026 reports)**

| | VTV | MFS |
|---|---|---|
| Has a DGTW benchmark | about 88% | about 77% |
| PERMNO, but not in DGTW | about 11% | about 21% |
| No PERMNO | about 1% | about 2% |

The missing names are not small: Chubb, Accenture, Linde and NXP are incorporated outside the U.S.; McKesson, Lowe's, Philip Morris and Marriott have negative book equity. Below, we compute CS, CT and AS on the covered stocks only, with their weights rescaled to sum to one. **Keep in mind that the benchmark speaks for only about three quarters of MFS.**

## Part 2. The characteristics view: what do these funds own?

Each benchmark cell has a three-digit code: size quintile, book-to-market quintile, momentum quintile (5 = largest, cheapest, strongest prior year). Summing a fund's weights by each digit shows its style from its *holdings*, not from a regression.

In [ ]:
def tilt(fund, report_dt):
    """Weight of the covered holdings in each size, B/M and momentum quintile."""
    h = coverage(fund, report_dt)
    h = h[h.status == "has a DGTW benchmark"]
    port = D[D.month == str(report_dt.to_period("M"))].set_index("permno")["port"].astype(str)
    w = h.groupby("permno").weight.sum()
    w, code = w / w.sum(), port.loc[w.index]
    return pd.DataFrame({name: w.groupby(code.str[k].values).sum() for k, name in enumerate(["size", "B/M", "momentum"])}).reindex(list("12345")).fillna(0)

fig, axes = plt.subplots(1, 3, figsize=(11, 3), sharey=True)
t = {f: tilt(f, H.loc[H.fund == f, "report_dt"].max()) for f in FUNDS}
for ax, col in zip(axes, ["size", "B/M", "momentum"]):
    pd.DataFrame({f: t[f][col] for f in FUNDS}).plot.bar(ax=ax, rot=0)
    ax.set_title(col); ax.set_xlabel("quintile (1 = low, 5 = high)")
axes[0].set_ylabel("share of covered weight")
plt.tight_layout(); plt.show()
pd.concat(t, axis=1).round(2)

**How to read the output (our run)**

- **Size:** both funds are almost entirely in the largest size quintile. These are large-cap funds, as advertised.
- **B/M:** VTV leans toward cheap stocks (about a third of its weight in the highest B/M quintile) but still has about a quarter in the two lowest. MFS is spread more evenly. Remember that the DGTW B/M is *industry-adjusted*: a bank is compared with other banks.
- **Momentum:** VTV is spread fairly evenly. MFS has little in either extreme.

**Connect to Exercise 2.** VTV's HML loading was well below 1. Here you see why from the holdings side: VTV owns the cheaper half of large caps, not only the extreme value stocks that HML is built from. And VTV's index ranks stocks on several valuation ratios, not B/M alone.

## Part 3. Compute CS, CT and AS

**Timing rule.** Each month we use the latest holdings report dated on or before the *end of the previous month*: what the fund held as the month began. If a month's report is missing, the previous one carries forward, which is what DGTW (1997) did with quarterly holdings. "A year ago" uses the same rule twelve months earlier.

`QUARTER_END_ONLY = True` keeps only March, June, September and December reports, as most studies had to. Try it after your first run.

In [ ]:
QUARTER_END_ONLY = False

cell = D.groupby(["month", "port"])["bench_ret"].first()    # each cell's return in each month
by_month = {m: g.set_index("permno") for m, g in D.groupby("month")}

def latest_report(fund, month):
    """Most recent report dated on or before the end of `month`."""
    dates = H.loc[(H.fund == fund) & (H.report_dt <= pd.Period(month, "M").to_timestamp(how="end")), "report_dt"]
    if QUARTER_END_ONLY:
        dates = dates[dates.dt.month.isin([3, 6, 9, 12])]
    return dates.max()

def stock_weights(fund, report_dt, month):
    """Weights of the holdings that have a benchmark in `month`, rescaled to sum to 1, and the share covered."""
    w = H[(H.fund == fund) & (H.report_dt == report_dt)].dropna(subset=["permno"]).groupby("permno")["weight"].sum()
    w = w[w.index.isin(by_month[month].index)]
    return w / w.sum(), w.sum()

def decompose(fund):
    rows, parts = [], []
    for m in pd.period_range(START, END, freq="M"):
        now, ago = str(m), str(m - 12)
        w, covered = stock_weights(fund, latest_report(fund, str(m - 1)), now)     # held as the month began
        w_ago, _ = stock_weights(fund, latest_report(fund, str(m - 13)), ago)      # held a year earlier
        d = by_month[now].loc[w.index]
        cell_ago = by_month[ago].loc[w_ago.index, "port"]                          # each stock's cell a year ago
        r_cell_ago = cell.loc[now].reindex(cell_ago.values).values                 # ... and that cell's return this month
        cs_j = w * (d["ret"] - d["bench_ret"])
        rows.append({"month": now, "CS": cs_j.sum(), "AS": (w_ago.values * r_cell_ago).sum(),
                     "cells_now": (w * d["bench_ret"]).sum(), "holdings_ret": (w * d["ret"]).sum(), "covered": covered})
        parts.append(pd.DataFrame({"month": now, "ticker": d["ticker"].values, "cs": cs_j.values}))
    out = pd.DataFrame(rows).set_index("month")
    out["CT"] = out["cells_now"] - out["AS"]
    return out, pd.concat(parts)

res, contrib = {}, {}
for f in FUNDS:
    res[f], contrib[f] = decompose(f)
    gap = (res[f][["CS", "CT", "AS"]].sum(axis=1) - res[f]["holdings_ret"]).abs().max()
    print(f"{f}: {len(res[f])} months | largest |CS + CT + AS - holdings return| = {gap:.1e}")
res["MFS"][["CS", "CT", "AS", "holdings_ret", "covered"]].tail()

**Check yourself.** CS + CT + AS equals the holdings return up to rounding (about 1e-17): the three parts are an exact split. Why does that have to be true? (Write out the sum of the three formulas.)

## Part 4. Results: skill, luck, or noise?

In [ ]:
rows = []
for f in FUNDS:
    r = res[f].join(R[R.fund == f].set_index("month")["ret"].rename("fund_ret"))
    for col in ["CS", "CT", "AS", "holdings_ret", "fund_ret"]:
        m, t = nw_mean(r[col].dropna())
        rows.append({"fund": f, "part": col, "% per month": 100 * m, "t (Newey-West)": t, "% per year": 1200 * m})
summary = pd.DataFrame(rows).set_index(["fund", "part"]).round(2)
summary

**How to read the output (our run, January 2017 to September 2026, 117 months)**

| | VTV | MFS |
|---|---|---|
| CS | -1.15% a year, t = -0.87 | -1.94% a year, t = -1.27 |
| CT | -0.27% a year | -0.56% a year |
| AS | 13.9% a year | 13.5% a year |

- **Style is almost everything.** AS is close to each fund's whole return. Both funds earned what large-cap value-style stocks earned.
- **VTV is the placebo.** An index fund has no stock-picking, so its true CS should be near zero. Its measured CS is -1.2% a year, not significant. That is the method's noise level: the benchmark cells do not match VTV's index exactly (and cover only about 88% of it).
- **MFS:** CS is negative, and not significant. No evidence of selectivity skill over these ten years, but also not strong evidence against it.

In [ ]:
# Cumulative CS: does any period stand out?
fig, ax = plt.subplots(figsize=(8, 3.2))
for f in FUNDS:
    (1 + res[f]["CS"]).cumprod().plot(ax=ax, label=f)
ax.axhline(1, color="grey", lw=0.8); ax.set_title("Cumulative characteristic selectivity (CS), growth of $1"); ax.legend(); ax.set_xlabel("")
plt.tight_layout(); plt.show()
print({f: f"{(res[f]['CS'] > 0).mean():.0%} of months positive, monthly s.d. {100 * res[f]['CS'].std():.2f}%" for f in FUNDS})

In [ ]:
# Which holdings drove CS? Sum each stock's contribution over all months.
for f in FUNDS:
    by_stock = contrib[f].groupby("ticker")["cs"].sum() * 100
    print(f"{f}: biggest contributors to CS, in total percentage points over the sample")
    print("  best :", by_stock.nlargest(5).round(2).to_dict())
    print("  worst:", by_stock.nsmallest(5).round(2).to_dict())

**How to read the output (our run).** VTV's biggest positive contributor is Micron (about +4.3 percentage points in total), our running example from Weeks 1 and 2: a stock in an index fund can beat its look-alikes by a lot. Over the whole sample, other holdings more than offset it. For MFS, JPMorgan helped most and Comcast hurt most.

**Your turn.** (1) Pick the worst contributor in MFS. Was it a bad pick, or a fine company that lagged its look-alike stocks? (2) Set `QUARTER_END_ONLY = True` in Part 3 and rerun Parts 3 and 4. Do the conclusions change?

## Part 5. Holdings return vs. what investors actually earned

CS, CT and AS describe the fund's *stock holdings*. Investors earn the fund's reported return, after fees, cash, trading costs, and trades between report dates.

In [ ]:
for f in FUNDS:
    r = res[f].join(R[R.fund == f].set_index("month")["ret"].rename("fund_ret")).dropna()
    gap = r["fund_ret"] - r["holdings_ret"]
    m, t = nw_mean(gap)
    print(f"{f}: fund return - holdings return = {1200 * m:.2f}% a year (t = {t:.2f}); "
          f"expense ratio {100 * info.loc[f, 'exp_ratio']:.2f}%; correlation {r['fund_ret'].corr(r['holdings_ret']):.3f}")

**How to read the output (our run).** VTV trails its holdings by about 0.15% a year; its expense ratio is 0.04%. MFS trails by about 0.8% a year; its expense ratio is 0.54%. The rest of the gap is cash, trading costs, the holdings the benchmark does not cover, and trades the monthly snapshots miss. The correlations (0.999 and 0.993) say the snapshots track the funds well.

## Part 6. Discuss in groups

1. VTV is an index fund, yet its CS is about -1.2% a year. Is that skill (or anti-skill)? What does it tell you about how large CS must be before you believe it?
2. MFS's CS has t = -1.27 over ten years. Can you conclude the managers lack skill? Roughly how many more years of data would a true CS of -2% a year need before t reached -2?
3. About a quarter of MFS is outside the benchmark: foreign-incorporated firms, REITs, and firms with negative book equity (often big buyers of their own stock). How could leaving them out bias CS, and in which direction?
4. In the Session 2 slides, DGTW credits anything beyond size, B/M and momentum as "skill." MFS is a quality-focused value manager. If profitable firms earned a premium over this period, where would that premium show up here?
5. Part 2 says VTV owns the cheaper half of large caps. Exercise 2 said its HML loading was about 0.37. Are these two answers to the same question, or to different questions?